# Tidy Data Practice

1) Each variable forms a column  
2) Each observation forms a row  
3) Each type of observational unit forms a table  

But really, we are doing normal forms practice as well

In [1]:
import pandas as pd

---

## Pew survey data

In [2]:
pew = pd.read_csv('Data/pew_research.csv') #if you don't have a data directory just delete Data/
pew.head()

,religion,<$10k,$10-20k,$20-30k,$30-40k,$40-50k,$50-75k,$75-100k,$100-150k,>150k,Don't know/refused
0,Agnostic,27,34,60,81,76,137,122,109,84,96
1,Atheist,12,27,37,52,35,70,73,59,74,76
2,Buddhist,27,21,30,34,33,58,62,39,53,54
3,Catholic,418,617,732,670,638,1116,949,792,633,1489
4,Don’t know/refused,15,14,15,11,10,35,21,17,18,116


**Is this data tidy?**  
**What can we do?**

In [3]:
#let's melt this
pew_melt = pew.melt(id_vars = 'religion', var_name = 'income', value_name = 'num_respondents')

In [6]:
pew_melt = pew_melt.reset_index(drop = True)
pew_melt.head()

,religion,income,num_respondents
0,Agnostic,<$10k,27
1,Atheist,<$10k,12
2,Buddhist,<$10k,27
3,Catholic,<$10k,418
4,Don’t know/refused,<$10k,15


In [7]:
# Output to .csv for additional processing
pew_melt.to_csv('Data/pew_tidy.csv', index=False)

In [13]:
#Turn it back into the original table by pivoting
pew_pivot = pew_melt.pivot(index = 'religion', columns = 'income', values = 'num_respondents')
pew_pivot = pew_pivot.reset_index()
pew_pivot.columns.name = None
pew_pivot.head()

,religion,$10-20k,$100-150k,$20-30k,$30-40k,$40-50k,$50-75k,$75-100k,<$10k,>150k,Don't know/refused
0,Agnostic,34,109,60,81,76,137,122,27,84,96
1,Atheist,27,59,37,52,35,70,73,12,74,76
2,Buddhist,21,39,30,34,33,58,62,27,53,54
3,Catholic,617,792,732,670,638,1116,949,418,633,1489
4,Don’t know/refused,14,17,15,11,10,35,21,15,18,116


In [15]:
pew_pivot.columns = ['religion', '<$10k', '$10-20k',  '$20-30k', '$30-40k', '$40-50k',
       '$50-75k', '$75-100k', '$100-150k','>150k', "Don't know/refused"]

In [16]:
pew_pivot.head()

,religion,<$10k,$10-20k,$20-30k,$30-40k,$40-50k,$50-75k,$75-100k,$100-150k,>150k,Don't know/refused
0,Agnostic,34,109,60,81,76,137,122,27,84,96
1,Atheist,27,59,37,52,35,70,73,12,74,76
2,Buddhist,21,39,30,34,33,58,62,27,53,54
3,Catholic,617,792,732,670,638,1116,949,418,633,1489
4,Don’t know/refused,14,17,15,11,10,35,21,15,18,116


## Coordinates - explode vs expand

In [17]:
cities = pd.read_csv('Data/world_cities_edit.csv') #if you don't have a Data directory just delete Data/

In [18]:
cities.head()

,city_ascii,coordinates,country,capital,population
0,Tokyo,"(35.685,139.7514)",Japan,primary,39105000.0
1,Jakarta,"(-6.1753,106.8269)",Indonesia,primary,33756000.0
2,Guangzhou,"(23.13,113.26)",China,admin,26940000.0
3,Mumbai,"(19.0758,72.8775)",India,admin,24973000.0
4,Shanghai,"(31.2325,121.4692)",China,admin,24870895.0


In [19]:
lat_lon = cities['coordinates'].str.replace('(','', regex = False).str.replace(')','', regex = False).str.split(',', expand = True)

In [20]:
lat_lon.head()

,0,1
0,35.685,139.7514
1,-6.1753,106.8269
2,23.13,113.26
3,19.0758,72.8775
4,31.2325,121.4692


In [21]:
#What do we do now?
lat_lon.columns = ['lat','lon']
lat_lon.head()

,lat,lon
0,35.685,139.7514
1,-6.1753,106.8269
2,23.13,113.26
3,19.0758,72.8775
4,31.2325,121.4692


In [25]:
cities_new = pd.concat([cities.drop(columns = 'coordinates'), lat_lon], axis = 1)

In [ ]:
#let's put them back in a new table

OK so what would explode do?

In [26]:
cities.explode(column = 'coordinates') #hmmm, what?

,city_ascii,coordinates,country,capital,population
0,Tokyo,"(35.685,139.7514)",Japan,primary,39105000.0
1,Jakarta,"(-6.1753,106.8269)",Indonesia,primary,33756000.0
2,Guangzhou,"(23.13,113.26)",China,admin,26940000.0
3,Mumbai,"(19.0758,72.8775)",India,admin,24973000.0
4,Shanghai,"(31.2325,121.4692)",China,admin,24870895.0
...,...,...,...,...,...
50245,Al Jabin,"(14.704,43.599)",Yemen,admin,NaN
50246,`Amran,"(15.6594,43.9439)",Yemen,admin,NaN
50247,Nelspruit,"(-25.4745,30.9703)",South Africa,admin,NaN
50248,Lupane,"(-18.9315,27.807)",Zimbabwe,admin,NaN


In [27]:
cities['coordinates'] = cities['coordinates'].str.replace('(','', regex = False).str.replace(')','', regex = False).str.split(',')

In [28]:
#try again with explode
cities.explode('coordinates')

,city_ascii,coordinates,country,capital,population
0,Tokyo,35.685,Japan,primary,39105000.0
0,Tokyo,139.7514,Japan,primary,39105000.0
1,Jakarta,-6.1753,Indonesia,primary,33756000.0
1,Jakarta,106.8269,Indonesia,primary,33756000.0
2,Guangzhou,23.13,China,admin,26940000.0
...,...,...,...,...,...
50247,Nelspruit,30.9703,South Africa,admin,NaN
50248,Lupane,-18.9315,Zimbabwe,admin,NaN
50248,Lupane,27.807,Zimbabwe,admin,NaN
50249,Charlotte Amalie,18.342,U.S. Virgin Islands,primary,NaN


---

## Ebola data - More complex - we may not get to this in class so you have all of it

This is more complicated than anything I will ask you to do in the homework.

In [ ]:
ebola = pd.read_csv('Data/country_timeseries.csv')
ebola.head(3)

### How can you melt this table?

In [ ]:
ebola_melt = ebola.melt(id_vars=['Date', 'Day'])
ebola_melt.head(3)

In [ ]:
# Split the 'variable' column into Country and another temp column
ebola_melt[['stat', 'Country']] = ebola_melt['variable'].str.split('_', expand=True)
ebola_melt.sort_values(by=['Country','Day']).head(4)

In [ ]:
#We can also do it this way
#ebola_melt['stat'] = ebola_melt['variable'].apply(lambda x: x.split('_')[0])
#ebola_melt['Country'] = ebola_melt['variable'].apply(lambda x: x.split('_')[1])

In [ ]:
ebola_melt.sort_values(by = ['Country', 'Day']).head()

In [ ]:
# a few things
#get rid of the combined variable
ebola_melt.drop(columns = 'variable', inplace = True)

In [ ]:
ebola_melt.sort_values(by = ['Country', 'Day']).head()

In [ ]:
#cases and deaths are two separate things, so lets pivot the table

The 👆 above data is NOT tidy! Check the first 2 rows. 

In [ ]:
ebola_pivot = ebola_melt.pivot_table(index=['Date', 'Day', 'Country'], # The column(s) that will stay the same
                                     columns='stat', # The column to pivot out
                                     values='value', # The data that goes in the new columns
                                     # dropna=False   # Set to False if you want to keep Null/NaN records
                                    ).reset_index()
ebola_pivot.sort_values(by=['Country','Day']).head(2)

Since Date <=> Day, move that relation to a separate table

In [ ]:
ebola_days = ebola_pivot[['Day', 'Date']].drop_duplicates()
ebola_days.head(3)

In [ ]:
ebola_days.to_csv('Data/ebola_tidy_days.csv', index=False)

In [ ]:
ebola_final = ebola_pivot.drop(columns='Date') 

In [ ]:
# Save to .csv
ebola_final.to_csv('Data/ebola_tidy_main.csv', index=False)